In [ ]:
import subprocess, os
from pathlib import Path

subprocess.run(["pip", "install", "-q", "--upgrade", "ultralytics"])

if not Path("/kaggle/working/repo").exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/No6Name8/Project_SAR_image.git",
        "/kaggle/working/repo"
    ])

import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DATASET_ROOT = Path("/kaggle/input/datasets/greatbird/sardet-100k/SARDet_100K")
print(f"Dataset found: {DATASET_ROOT.exists()}")
print(f"Contents     : {[p.name for p in DATASET_ROOT.iterdir()]}")

In [ ]:
import subprocess

subprocess.run([
    "python", "/kaggle/working/repo/coco_to_yolo.py",
    "--coco-json",  str(DATASET_ROOT / "Annotations/train.json"),
    "--images-dir", str(DATASET_ROOT / "JPEGImages/train"),
    "--labels-dir", "/kaggle/working/labels/train"
])

subprocess.run([
    "python", "/kaggle/working/repo/coco_to_yolo.py",
    "--coco-json",  str(DATASET_ROOT / "Annotations/val.json"),
    "--images-dir", str(DATASET_ROOT / "JPEGImages/val"),
    "--labels-dir", "/kaggle/working/labels/val"
])

In [ ]:
yaml_content = f"""
path: {DATASET_ROOT}
train: /kaggle/working/labels/train
val: /kaggle/working/labels/val

nc: 6
names:
  0: ship
  1: aircraft
  2: car
  3: tank
  4: bridge
  5: harbor
"""
with open("/kaggle/working/sardet_kaggle.yaml", "w") as f:
    f.write(yaml_content)

from ultralytics import YOLO

model = YOLO("yolov12n.pt")
results = model.train(
    data="/kaggle/working/sardet_kaggle.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    project="/kaggle/working/results",
    name="yolov12n_full",
)

In [ ]:
import pandas as pd

metrics = model.val(
    data="/kaggle/working/sardet_kaggle.yaml",
    imgsz=640,
    device=0,
)

results_dict = {
    "model": ["yolov12n"],
    "params_M": [2.4],
    "mAP50": [round(metrics.box.map50, 3)],
    "mAP75": [round(metrics.box.map75, 3)],
    "mAP50_95": [round(metrics.box.map, 3)],
    "AP_ship":     [round(metrics.box.aps[0], 3)],
    "AP_aircraft": [round(metrics.box.aps[1], 3)],
    "AP_car":      [round(metrics.box.aps[2], 3)],
    "AP_tank":     [round(metrics.box.aps[3], 3)],
    "AP_bridge":   [round(metrics.box.aps[4], 3)],
    "AP_harbor":   [round(metrics.box.aps[5], 3)],
}

import pandas as pd
df = pd.DataFrame(results_dict)
print(df)
df.to_csv("/kaggle/working/results/baseline_results.csv", index=False)